# AutoKernel H100 Workflow

This notebook is the cloud-H100 entrypoint for the AutoKernel scaffold. Run the environment checks first, then update the model configuration cell when you are ready to provide the real model.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
print(f"repo: {ROOT}")
print(f"python: {sys.version.split()[0]}")

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print(f"torch installed: {has_module('torch')}")
print(f"triton installed: {has_module('triton')}")

try:
    import torch
    print(f"torch version: {torch.__version__}")
    print(f"cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"device: {torch.cuda.get_device_name(0)}")
        print(f"capability: {torch.cuda.get_device_capability(0)}")
except Exception as exc:
    print(f"torch check failed: {type(exc).__name__}: {exc}")

nvidia_smi = subprocess.run(["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi -L || true"], text=True, capture_output=True)
print(nvidia_smi.stdout.strip() or "nvidia-smi not found")

## Install And Prepare

Use `uv sync` on the H100 host so PyTorch and Triton match the GPU runtime. The CUDA dependency set is the intended path for the paper-style benchmark loop.

In [ ]:
# Run once per fresh cloud machine.
# If uv is not installed yet:
#   !curl -LsSf https://astral.sh/uv/install.sh | sh

!uv sync --extra cuda --extra models
!uv run prepare.py

## Optional Agent API Key

The profiling and benchmark scripts do not need an API key. Only set this if you will run an AI coding agent inside the cloud notebook or VM.

In [ ]:
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    key = getpass.getpass("OPENAI_API_KEY, leave blank to skip: ")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY is set for this notebook kernel.")
    else:
        print("Skipped. You can still profile, extract, benchmark, and verify kernels.")
else:
    print("OPENAI_API_KEY is already set for this notebook kernel.")

## Model Configuration

Keep this as the toy model until the real model arrives. For a local model, set `model`; for a HuggingFace model, set `module` and `pretrained`.

In [ ]:
CONFIG = {
    "model": "models/llama_7b.py",
    "module": None,
    "class_name": "LlamaModel",
    "pretrained": None,
    "input_shape": "1,512",
    "dtype": "float16",
    "top": 5,
    "backend": "triton",
}

CONFIG

In [ ]:
def run(cmd: list[str], *, log: str | None = None) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd))
    if log:
        with open(log, "w", encoding="utf-8") as f:
            proc = subprocess.run(cmd, text=True, stdout=f, stderr=subprocess.STDOUT)
        print(f"wrote {log}; exit={proc.returncode}")
        if proc.returncode != 0:
            print(Path(log).read_text(encoding="utf-8")[-4000:])
        return proc
    proc = subprocess.run(cmd, text=True)
    print(f"exit={proc.returncode}")
    return proc

def model_args() -> list[str]:
    args = []
    if CONFIG["model"]:
        args.extend(["--model", CONFIG["model"]])
    if CONFIG["module"]:
        args.extend(["--module", CONFIG["module"]])
    args.extend(["--class-name", CONFIG["class_name"]])
    if CONFIG["pretrained"]:
        args.extend(["--pretrained", CONFIG["pretrained"]])
    args.extend(["--input-shape", CONFIG["input_shape"], "--dtype", CONFIG["dtype"]])
    return args

## Phase A: Profile And Extract

In [ ]:
run(["uv", "run", "profile.py", *model_args()], log="profile.log")
!python -m json.tool workspace/profile_report.json | head -80

run(["uv", "run", "extract.py", "--top", str(CONFIG["top"]), "--backend", CONFIG["backend"]])
!find workspace -maxdepth 2 -type f | sort

## Phase B: Benchmark Loop Smoke Test

In [ ]:
run(["uv", "run", "bench.py", "--quick"], log="run.log")
!grep -E "correctness|throughput_tflops|latency_us|pct_peak|bottleneck|speedup_vs_pytorch|peak_vram_mb" run.log || tail -80 run.log

run(["uv", "run", "orchestrate.py", "plan"])
run(["uv", "run", "orchestrate.py", "next"])

## Phase C: End-To-End Verification

In [ ]:
run(["uv", "run", "verify.py", *model_args()], log="verify.log")
!tail -120 verify.log